# Spiking ResNet on the PYNQ-Z2: live demo

**What you will see (in one picture):**
* **left**: the CIFAR-10 test image, plus the board's LEDs
* **middle**: a *spike raster*, one block of rows per layer, dots appear as neurons fire over the 32 timesteps
* **right**: live class bars (share of output spikes) and a gauge for the early-exit rule

**What is real:** every timestep, the output spikes are written to *your circuit on the FPGA* (`ee_axil`). The FPGA decides when to stop.

**What you must know / say honestly:**
* **LIVE** mode: the ARM CPU computes all layers while you watch (~0.5 s per timestep).
* **REPLAY** mode: spike trains were computed earlier and are replayed at a chosen speed. The FPGA decision is still live, but the spikes are *not* computed live. The dashboard prints which mode is running.
* "Confidence" bars = *share of output spikes so far*. They are **not** calibrated probabilities.
* The 4 LEDs and the first-spike LED only work with the **LED bitstream (v3)**. With the older bitstream you get a *virtual* LED panel on the screen.

Run the cells top to bottom. Cell 1 is the only one you may need to edit.

In [ ]:
# ---- CELL 1: settings (edit if needed) ----
import os, sys, time, numpy as np
BASE = os.environ.get("SNN_BASE", "/home/xilinx/jupyter_notebooks/SNN")   # folder where you unzipped snn_board_files.zip
ROOT, BIT, CIFAR = BASE + "/weights", BASE + "/snn.bit", BASE + "/cifar"
ENCODING, NORM = "poisson", "unit"      # use the flags that gave ~77% on Day 1
T_MIN, MARGIN = 4, 10                   # your chosen early-exit settings
CACHE = BASE + "/demo_cache.npz"        # used by REPLAY mode (made in the optional cell below)
USE_MOCK = os.environ.get("SNN_MOCK") == "1"   # software stand-in for the FPGA (testing without a board only)
sys.path[:0] = [BASE + "/golden", BASE + "/board"]
print("settings ok; mock FPGA:", USE_MOCK)

In [ ]:
# ---- CELL 2: load the network, the test images, and connect to your FPGA circuit ----
import snn_golden as g, live_dashboard as L
from fpga_ee import FpgaEE, MockMMIO, open_overlay
scale = 1 / 64 if ENCODING == "direct" else 1.0
net = g.Net(g.Params(ROOT, scale), "fixed", scale)
X, Y = g.load_cifar_test(CIFAR, 500)
MM = MockMMIO() if USE_MOCK else open_overlay(BIT)
probe = FpgaEE(MM, T_MIN, MARGIN)
print("FPGA circuit found (magic OK).  LED build (v3) loaded:", probe.has_extras)
print("images available:", len(X))

In [ ]:
# ---- CELL 3: LED self-test (only does something with the LED bitstream) ----
# Shows every class code 0..9 on LD0..LD3 for a moment, then turns them off.
if probe.has_extras:
    for c in range(10):
        probe.set_leds(c); print("class", c, L.CLASSES[c], format(c, "04b"), end="   \r"); time.sleep(0.35)
    probe.set_leds(0); print("\nLED self-test done")
else:
    print("This bitstream has no LED support (v2): the dashboard will show a virtual LED panel instead.")

In [ ]:
# ---- CELL 4: the demo function ----
def demo(idx=1, mode="LIVE", margin=MARGIN, t_min=T_MIN, speed=0.3, leds=True, inplace=True):
    fpga = FpgaEE(MM, t_min, margin)                    # (re)configures the FPGA rule: t_min, margin
    show = L.jupyter_display(inplace=inplace)
    label = "CIFAR-10 test #%d (true: %s)" % (idx, L.CLASSES[Y[idx]])
    if mode == "LIVE":
        img = (X[idx] * 255).astype(np.uint8)
        it = L.stream_full(net, X[idx], 32, ENCODING, NORM, seed=idx)
        return L.run_image(img, it, fpga, 32, show, "LIVE", 0.0, leds, label)
    if not os.path.exists(CACHE):
        print("REPLAY needs the cache: run the optional cell below once."); return None
    c = np.load(CACHE); ids = list(c["idxs"])
    if idx not in ids:
        print("image %d is not in the cache. Cached images: %s" % (idx, ids)); return None
    i = ids.index(idx)
    return L.run_image(c["imgs"][i], L.replay_iter(c, i), fpga, 32, show, "REPLAY", speed, leds, label)
print("demo() ready")

In [ ]:
# ---- CELL 5 (OPTIONAL, run once): pre-compute spike trains so REPLAY mode is instant ----
# About 16 s per image on the board's ARM. Choose images that look good in the demo.
IDXS = [0, 1, 2, 3, 4, 5, 6, 7]
L.make_cache(net, X, Y, IDXS, 32, ENCODING, NORM, CACHE)
print("cache written:", CACHE)

In [ ]:
# ---- CELL 6: interactive controls ----
try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    w_idx = W.IntSlider(value=1, min=0, max=len(X) - 1, description="image", continuous_update=False)
    w_mode = W.ToggleButtons(options=["LIVE", "REPLAY"], description="mode")
    w_margin = W.IntSlider(value=MARGIN, min=3, max=15, description="margin")
    w_tmin = W.IntSlider(value=T_MIN, min=1, max=10, description="t_min")
    w_speed = W.FloatSlider(value=0.3, min=0.05, max=1.0, step=0.05, description="replay s/step")
    w_led = W.Checkbox(value=True, description="drive board LEDs")
    w_btn = W.Button(description="RUN", button_style="success", icon="play")
    w_out = W.Output()
    def on_run(_):
        with w_out:
            clear_output(wait=True)
            demo(w_idx.value, w_mode.value, w_margin.value, w_tmin.value, w_speed.value, w_led.value, inplace=False)
    w_btn.on_click(on_run)
    display(W.VBox([W.HBox([w_idx, w_mode]), W.HBox([w_margin, w_tmin, w_speed]), w_led, w_btn, w_out]))
except ImportError:
    print("ipywidgets is not installed - use the plain cell below instead.")

In [ ]:
# ---- CELL 7: plain run (no widgets needed): one image, LIVE ----
result = demo(idx=1, mode="LIVE")
print(result)